This notebook runs segmentation using the tensorstore package, and the associated changed made to the neurotorch array and predictor classes.

In [1]:
import tracemalloc
from datetime import datetime
import torch
from matplotlib import pyplot as plt
import os
import ac_segmentation
import ac_segmentation.neurotorch.datasets.dataset
from ac_segmentation.neurotorch.datasets.dataset import open_ZarrTensor, create_EmptyTensor
import ac_segmentation.neurotorch.core.predictor
import ac_segmentation.neurotorch.nets.RSUNet

Predictor = ac_segmentation.neurotorch.core.predictor.Predictor
Vector = ac_segmentation.neurotorch.datasets.datatypes.Vector
BoundingBox = ac_segmentation.neurotorch.datasets.datatypes.BoundingBox
TSArray = ac_segmentation.neurotorch.datasets.dataset.TSArray

In [2]:
##Open input tensor
in_arr = open_ZarrTensor('/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos89/1/', bytes_limit= 100_000_000)
in_arr = in_arr[0,0,:,:,:].transpose()

##Create output tensor
out_arr = create_EmptyTensor('/ACdata/Users/connorl/Out_Array.zarr', in_arr.shape, dtype = 'int16')

In [ ]:
start = datetime.now()

checkpt_file = "/allen/programs/celltypes/workgroups/mousecelltypes/MachineLearning/Olga/forConnor/mip1_model/best.ckpt"
net = ac_segmentation.neurotorch.nets.RSUNet.RSUNet()
predictor = Predictor(net, checkpt_file, gpu_device=None)

##Create input and output array objects
inarr = TSArray(in_arr, iteration_size=BoundingBox(Vector(0, 0, 0),Vector(32, 32, 32)), stride=Vector(64, 64, 64))
outarr = TSArray(out_arr, prob_map=True)

##Run segmentation
torch.set_num_threads(56) 
predictor.run(inarr, outarr, batch_size=100, max_pix = 30000)

end = datetime.now()
print(end-start)

In [ ]:
plt.imshow(a[140, :, 12000:13000])